# ResNet Architecture

Residual Networks (ResNet) are deep neural network architectures designed to enable the training of very deep models without suffering from the **degradation problem**, where deeper networks begin to perform worse than shallower ones. The key idea is to allow layers to learn **residual mappings** instead of directly learning the full transformation.

---

## 1. Intuition

As neural networks become deeper, we would expect them to perform better because they can represent more complex functions. However, in practice, very deep networks often perform **worse** than shallower ones. This phenomenon is not primarily caused by overfitting but by **optimization difficulties**.

When many layers are stacked, the signal must pass through repeated nonlinear transformations. During backpropagation, gradients may vanish or explode, making it difficult for early layers to learn.

ResNet introduces the concept of **skip connections** (also called **identity shortcuts**) to address this problem.

Instead of forcing a set of layers to learn a direct mapping:

$$
H(x)
$$

the network learns a **residual function**:

$$
F(x) = H(x) - x
$$

The original mapping can then be written as:

$$
H(x) = F(x) + x
$$

This means the network only needs to learn the **difference** between the desired mapping and the identity mapping.

If the optimal transformation is close to identity, the residual \(F(x)\) will be small, making the optimization easier.

A **Residual Block** therefore looks like:

input x <br />
│ <br />
├── Conv <br />
├── BatchNorm <br />
├── ReLU <br />
├── Conv <br />
├── BatchNorm <br />
│ <br />
└───────(+ identity shortcut) <br />
│ <br />
ReLU


The shortcut connection allows information and gradients to flow directly through the network, enabling the training of networks with **hundreds of layers**.

---

## 2. Mathematical Formulation

Consider an input vector \(x\) entering a block composed of several nonlinear transformations (e.g., convolution, batch normalization, activation). In a traditional neural network layer, the output would be:

$$
y = H(x)
$$

where $H(x)$ represents the learned mapping.

In a residual block, the output becomes:

$$
y = F(x, W) + x
$$

where:

- $F(x, W)$ is the **residual function** represented by the stacked layers with parameters \(W\)
- $x$ is the identity shortcut connection

For example, if the residual block contains two convolutional layers:

$$
F(x) = W_2 \sigma(W_1 x)
$$

where

- $W_1, W_2$ are convolutional operators
- $\sigma$ is a nonlinear activation (typically ReLU)

Thus the block output is:

$$
y = W_2 \sigma(W_1 x) + x
$$

---

### Backpropagation through Residual Connections

One of the most important properties of ResNet appears when computing gradients.

Suppose the loss function is $L$. The gradient with respect to the input $x$ is:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\left(
\frac{\partial F(x)}{\partial x} + I
\right)
$$

The presence of the **identity matrix $I$** ensures that gradients can propagate directly through the shortcut path.

Even if

$$
\frac{\partial F(x)}{\partial x}
$$

becomes very small (vanishing gradient), the identity term guarantees that gradients still flow backward.

This mechanism is what allows **very deep networks (50, 101, 152 layers)** to remain trainable.

---

## 3. Geometric Interpretation

From a geometric perspective, deep neural networks perform **progressive transformations of the feature space**.

Each layer applies a nonlinear transformation that reshapes the data manifold in order to make classes more separable.

In a traditional deep network:

$$
x_{l+1} = H(x_l)
$$

the transformation between layers can be large and difficult to optimize. The network must learn a completely new representation at each stage.

In a residual network:

$$
x_{l+1} = x_l + F(x_l)
$$

the transformation becomes **incremental**.

Each residual block performs only a **small displacement in representation space**.

This can be interpreted as a **discrete dynamical system**:

$$
x_{l+1} = x_l + F(x_l)
$$

which closely resembles the **Euler discretization of an ordinary differential equation**:

$$
\frac{dx}{dt} = F(x)
$$

Under this view:

- The network depth corresponds to **time steps**
- Each residual block applies a **small update to the representation**

Rather than learning entirely new feature spaces at each layer, the network **iteratively refines the representation**.

This perspective helps explain why residual networks are easier to optimize: the network learns **smooth trajectories in feature space** rather than abrupt transformations.

## Example: Classify Images Containing Three Geometric Shapes

In this experiment we construct and train a **Convolutional Neural Network (CNN)** to classify synthetic grayscale images containing one of three geometric shapes: **square, circle, or triangle**. The images are generated programmatically and have a resolution of **32×32 pixels** with a single channel.

Synthetic datasets are useful for studying neural networks because they allow us to **control the data generation process**, isolate specific visual patterns, and better understand how convolutional networks learn representations such as edges, corners, and curved boundaries.

To make the task more realistic and prevent the model from simply memorizing pixel patterns, we introduce two types of **data augmentation** during dataset generation:

- **Random rotation:** Shapes are rotated by random angles so that the network must learn **orientation-invariant features** rather than fixed spatial templates.
- **Additive noise:** Random noise is added to the images, forcing the model to learn **robust visual features** rather than relying on perfectly clean edges.

These transformations encourage the CNN to learn meaningful geometric features such as **edges, diagonals, and curvature**, which are shared across different shapes.

---

### Motivation

The goal of this experiment is twofold:

1. **Demonstrate how convolutional neural networks learn geometric features** from raw pixel data.
2. **Illustrate the mechanics of a residual architecture (ResNet)** in a simple and interpretable setting.

Residual networks introduce **skip connections** that allow the network to learn a residual mapping of the form:

$$
H(x) = F(x) + x
$$

Instead of learning a direct transformation \(H(x)\), the network learns the residual function \(F(x)\), which often makes optimization easier and improves gradient propagation in deeper networks.

---

### Proposed Architecture

The model follows a simplified **ResNet-style architecture** consisting of a convolutional feature extractor, a residual connection, and a lightweight classifier.

input (1×32×32) 

conv3×3 <br /> 
batchnorm <br />
ReLU 

conv3×3 <br /> 
batchnorm <br /> 
\+ <br /> 
skip connection<br /> 
ReLU 


maxpool 2×2 <br /> 
global average pooling <br /> 
linear → 3

In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

rng = np.random.default_rng(42)
torch.manual_seed(42)

g = torch.Generator()
g.manual_seed(42)

def make_rnd_square(rng):
    img = np.zeros((32,32), dtype=np.float32)

    k = rng.integers(low=5, high=13)
    x = rng.integers(0, 32-k)
    y = rng.integers(0, 32-k)
    img[x : x+k, y : y+k] = 1

    return img

squares = np.array([make_rnd_square(rng) for i in range(1000)])

plt.imshow(squares[0], cmap="gray")
plt.title("Random Square")
plt.axis("off")
plt.show()

ModuleNotFoundError: No module named 'torch'

In [ ]:
def draw_rnd_circle(rng):

    img = np.zeros((32, 32), dtype=np.float32)
    r = rng.integers(low=5, high=13)
    cx = rng.integers(r, 32-r)
    cy = rng.integers(r, 32-r)

    for i in range(32):
        for j in range(32):
            if (i-cx)**2 + (j-cy)**2 <= r**2:
                img[i,j] = 1

    return img


circles = np.array([draw_rnd_circle(rng) for i in range(1000)])

plt.imshow(circles[0], cmap="gray")
plt.title("Random Circle")
plt.axis("off")
plt.show()

In [ ]:

def draw_rnd_isosceles_triangle(rng):

    img = np.zeros((32, 32), dtype=np.float32)
    k = rng.integers(5, 12)

    top_x = rng.integers(0, 32 - k)
    center_y = rng.integers(k, 32 - k)

    for d in range(k):
        row = top_x + d
        left = center_y - d
        right = center_y + d

        img[row, left:right+1] = 1
    
    return img

triangles = np.array([draw_rnd_isosceles_triangle(rng) for i in range(1000)])

plt.imshow(triangles[0], cmap="gray")
plt.title("Random Triangles")
plt.axis("off")
plt.show()

In [ ]:

def add_noise(rng, img):
    positions = [(rng.integers(0, 32), rng.integers(0, 32)) for i in range(30)]

    for p in positions:
        img[p[0], p[1]] = 1
    
    return img

def move(rng, img):
    aux = rng.integers(low=0, high=6)
    
    if aux == 0:
        img = img.T
    elif aux == 1:
        img = img.T * -1
    elif aux == 3:
        img = np.flip(img)
    elif aux == 4:
        img = np.rot90(img)
    elif aux == 5:
        img = np.rot90(img.T)
    
    return img


In [ ]:
x = np.vstack((squares, circles, triangles))
x = np.array([add_noise(rng, x[i]) for i in range(len(x))])

idxs = rng.integers(low=0, high=3000, size=1500)

for idx in idxs:
    x [idx]= move(rng, x[idx])

y = np.hstack(([0.] * len(squares), [1.] * len(circles), [2.] * len(triangles))).reshape(-1, 1)

permutation = [int(i) for i in rng.permutation(len(x))]
x_permuted = x[permutation]
y_permuted = y[permutation] 

# Adding a color channel
x_permuted = x_permuted[:, None, :, :]

X_torch = torch.tensor(x_permuted, dtype=torch.float32)
y_torch = torch.tensor(y_permuted, dtype=torch.float32)

plt.imshow(x[idxs[0]], cmap="gray")
plt.title("Random Figure")
plt.axis("off")
plt.show()

In [ ]:

class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if in_channels != out_channels:
            self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.projection = nn.Identity()

    def forward(self, x):

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + self.projection(x)
        out = F.relu(out)

        return out
        

In [ ]:
class SimpleResNet(nn.Module):

    def __init__(self, channels=8, num_classes=3):
        super().__init__()

        self.conv_in = nn.Conv2d(1, channels, kernel_size=3, padding=1)
        self.bn_in = nn.BatchNorm2d(channels)

        self.res_block = ResidualBlock(channels, channels)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):

        x = self.conv_in(x)
        x = self.bn_in(x)
        x = F.relu(x)

        x = self.res_block(x)
        x = self.pool(x)
        x = self.global_pool(x)

        x = torch.flatten(x, 1)
        x = self.fc(x)

        if self.training == False:
             x = F.softmax(x, dim=1)

        return x

In [ ]:

dataset = TensorDataset(X_torch, y_torch)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    generator=g
)

model = SimpleResNet(channels=8)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

for epoch in range(30):
    for X_batch, y_batch in loader:

        optimizer.zero_grad()
        outputs = model(X_batch)

        y_batch = y_batch.squeeze().long()
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

    if epoch % 5 == 0 or epoch == 29:
        print("epoch:", epoch, "loss:", loss.item())

In [ ]:
squares = np.array([make_rnd_square(rng) for i in range(500)])
circles = np.array([draw_rnd_circle(rng) for i in range(500)])
triangles = np.array([draw_rnd_isosceles_triangle(rng) for i in range(500)])

x = np.vstack((squares, circles, triangles))
x = np.array([add_noise(rng, x[i]) for i in range(len(x))])

idxs = rng.integers(low=0, high=len(x), size=1500)

for idx in idxs:
    x [idx]= move(rng, x[idx])

y = np.hstack(([0.] * len(squares), [1.] * len(circles), [2.] * len(triangles))).reshape(-1, 1)

permutation = [int(i) for i in rng.permutation(len(x))]
x_permuted = x[permutation]
y_permuted = y[permutation] 

# Adding a color channel
x_permuted = x_permuted[:, None, :, :]

torch.manual_seed(42)
X_torch = torch.tensor(x_permuted, dtype=torch.float32)
y_torch = torch.tensor(y_permuted, dtype=torch.float32)

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_torch)
    preds = outputs.argmax(dim=1)

y_true = y_torch.cpu().numpy()
y_pred = preds.cpu().numpy()

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nPrecision:", precision_score(y_true, y_pred, average=None))
print("Recall:", recall_score(y_true, y_pred, average=None))
print("F1-score:", f1_score(y_true, y_pred, average=None))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nFull Classification Report:")
print(classification_report(y_true, y_pred))